%md
## Ingest Fact Data from Silver to Gold Layer

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, FloatType,DecimalType, IntegerType
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/yklynk@gmail.com/Azure_databricks_data_engineering_project_shopvista_ecomm/1_setup/utilities

In [0]:
# Widget
dbutils.widgets.text('catalog', 'shopvista', 'catalog')
dbutils.widgets.text('storage_account_name', 'storageshopvista', 'storage_account_name')
dbutils.widgets.text('container_name', 'shopvista-raw-data', 'container_name')

In [0]:
catalog = dbutils.widgets.get('catalog')
storage_account_name = dbutils.widgets.get('storage_account_name')
container_name = dbutils.widgets.get('container_name')
print(catalog, storage_account_name, container_name)

%md
## ORDER ITEMS

In [0]:
df_silver= spark.readStream\
.format('delta')\
.option("checkpointLocation", "/Volumes/shopvista/raw/checkpoints/silver")\
.table(f'{catalog}.{silver_schema}.order_items')

In [0]:
from pyspark.sql.types import DecimalType, IntegerType

df_silver = df_silver.withColumn("quantity", F.col("quantity").cast(IntegerType())) \
                      .withColumn("unit_price", F.col("unit_price").cast(DecimalType(18, 2)))

In [0]:
# Create a new column named gross_amount
df_gold = df_silver.withColumn('gross_amount', F.col('quantity') * F.col('unit_price'))

#display(df_gold.limit(5))

In [0]:
#  Add discount_amount (discount_pct is already numeric, e.g., 21 -> 21%)
df_gold = df_gold.withColumn('discount_amount', F.col('gross_amount') * F.col('discount_pct')/100)

#display(df_silver.limit(5))


In [0]:
# Add sale_amount = gross - discount
df_gold = df_gold.withColumn('sales_amount', F.col('gross_amount')-F.col('discount_amount')+F.col("tax_amount"))

#display(df_gold.limit(5))


In [0]:
# add date id
df_gold = df_gold.withColumn("date_id", F.date_format(F.col("dt"), "yyyyMMdd").cast(IntegerType()))  # Create date_key

#display(df_gold.limit(5))



In [0]:
# Coupon flag
#  coupon flag = 1 if coupon_code is not null else 0
df_gold = df_gold.withColumn(
    "coupon_flag",
    F.when(F.col("coupon_code").isNotNull(), F.lit(1))
     .otherwise(F.lit(0))
)

#df_gold.limit(20).display()

In [0]:
df_gold = df_gold.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("order_id").alias("transaction_id"),
    F.col("customer_id"),
    F.col("item_seq").alias("seq_no"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("discount_amount"),
    F.col("tax_amount"),
    F.col("sales_amount").alias("net_amount")
)

### Write to Gold Table

In [0]:
gold_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/gold/fact_order_items/"
print(gold_checkpoint_path)

def upsert_to_gold(microBatchDF, batchId):
    table_name = f"{catalog}.gold.gld_fact_order_items"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("gold_table").merge(
            microBatchDF.alias("batch_table"),
            "gold_table.transaction_id = batch_table.transaction_id AND gold_table.seq_no = batch_table.seq_no",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

df_gold.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_gold
).format("delta").option("checkpointLocation", gold_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()

%md
## ORDER RETURNS

- Data Transformation & Encrichement

In [0]:
df_silver= spark.readStream\
.format('delta')\
.option("checkpointLocation", "/Volumes/shopvista/raw/checkpoints/silver")\
.table(f'{catalog}.{silver_schema}.order_returns')

In [0]:
#Add `date_id` column: yyyyMMdd format from order_dt
df_silver = df_silver.withColumn(
    "date_id",
    F.date_format(F.col("order_dt"), "yyyyMMdd")
)
#display(df_silver.limit(5))

In [0]:
#Calculate `return_days` = difference in days between return_ts and order_dt.
df_silver = df_silver.withColumn(
    "return_days",
    F.datediff(F.col("return_ts"), F.col("order_dt"))
)
#display(df_silver.limit(5))

In [0]:
#Create policy compliance flags: within_policy and is_late_return
df_silver = df_silver.withColumn(
    "within_policy",
    F.when(F.col("return_days") <= 15, 1).otherwise(0)
).withColumn(
    "is_late_return",
    F.when(F.col("return_days") > 15, 1).otherwise(0)
)
#display(df_silver.limit(5))

### Write to Gold Table

In [0]:
gold_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/gold/fact_order_returns/"
print(gold_checkpoint_path)

def upsert_to_gold(microBatchDF, batchId):
    table_name = f"{catalog}.gold.gld_fact_order_returns"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("gold_table").merge(
            microBatchDF.alias("batch_table"),
            "gold_table.transaction_id = batch_table.transaction_id AND gold_table.seq_no = batch_table.seq_no",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

df_gold.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_gold
).format("delta").option("checkpointLocation", gold_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()

## ORDER SHIPMENTS

In [0]:
df_silver= spark.readStream\
.format('delta')\
.option("checkpointLocation", "/Volumes/shopvista/raw/checkpoints/silver")\
.table(f'{catalog}.{silver_schema}.order_shipments')

In [0]:
#Add `carrier_group`: Domestic: ECOMEXPRESS, DELHIVERY, XPRESSBEES, BLUEDART. o International: All other carriers.
df_silver = df_silver.withColumn(
    "carrier_group",
    F.when(
        F.col("carrier").isin("ECOMEXPRESS", "DELHIVERY", "XPRESSBEES", "BLUEDART"),
        "Domestic"
    ).otherwise("International")
)
#display(df_silver.limit(5))

In [0]:
#Add `is_weekend_shipment` flag: True if `order_dt` is Saturday or Sunday, else False. 
df_silver = df_silver.withColumn(
    "is_weekend_shipment",
    F.dayofweek("order_dt").isin([7, 1])
)
#display(df_silver.limit(5))

In [0]:
gold_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/gold/fact_order_shipments"
print(gold_checkpoint_path)

def upsert_to_gold(microBatchDF, batchId):
    table_name = f"{catalog}.gold.gld_fact_order_shipments"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("gold_table").merge(
            microBatchDF.alias("batch_table"),
            "gold_table.transaction_id = batch_table.transaction_id AND gold_table.seq_no = batch_table.seq_no",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

df_gold.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_gold
).format("delta").option("checkpointLocation", gold_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()